# PVGIS-only ST-GNN — Monaco neural-SDE (drift/diffusion) pipeline

Reproducible orchestrator for the final run on server **newzealand**.

Does **not** duplicate runner/analysis logic — it builds commands and reads the
CSVs they write, via `physiq_pv.experiments.sde_pipeline`.

Safety switches: `RUN_TRAINING`, `RUN_ANALYSIS`, `RUN_SWEEP`, `CREATE_SWEEP`, `LOG_TO_WANDB`.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO_ROOT)

from physiq_pv.experiments import sde_pipeline as pipe
print('repo root:', REPO_ROOT)

In [ ]:
PVGIS_DIR = pipe.PVGIS_DIR
TEST_ANOMALY_SCORES = pipe.TEST_ANOMALY_SCORES
TRAIN_ANOMALY_SCORES = pipe.TRAIN_ANOMALY_SCORES
ANALYSIS_SCRIPT = pipe.ANALYSIS_SCRIPT

checks = {
    'PVGIS dir': Path(PVGIS_DIR).is_dir(),
    'test anomaly scores': Path(TEST_ANOMALY_SCORES).exists(),
    'train anomaly scores': Path(TRAIN_ANOMALY_SCORES).exists(),
    'analysis script': Path(ANALYSIS_SCRIPT).exists(),
}
try:
    import physiq_pv.experiments.pvgis_stgnn_runner  # noqa: F401
    checks['runner importable'] = True
except Exception as e:
    checks['runner importable'] = False
    print('runner import error:', e)
for k, v in checks.items():
    print(('OK     ' if v else 'MISSING') + '  ' + k)

If the anomaly scores are **missing**, regenerate them (run in a terminal — not launched automatically):

In [ ]:
if not (Path(TEST_ANOMALY_SCORES).exists() and Path(TRAIN_ANOMALY_SCORES).exists()):
    base = ('PYTHONPATH=$PWD python scripts/run_pvgis_climatology_anomaly_years.py'
            ' --pvgis-dir ' + PVGIS_DIR +
            ' --climatology-start-year 2005 --climatology-end-year 2023'
            ' --quantile 0.975 --climatology-window-days 15 --min-climatology-years 3'
            ' --variables solar_irradiance_poa pv_power_output temperature_2m wind_speed_10m'
            ' --out-root outputs')
    print('# Test year 2019:')
    print(base + ' --years 2019')
    print()
    print('# Train years 2016-2018 (aggregated):')
    print(base + ' --years 2016,2017,2018 --aggregate-out-dir ' + str(Path(TRAIN_ANOMALY_SCORES).parent))
else:
    print('anomaly scores present.')

## 2. Single-run config

In [ ]:
# === EXPERIMENT SELECTOR ===
# beta_nll   = Kong-faithful (train on ALL cells) + beta-NLL (Seitzer 2022,
#              beta_nll=0.5 from DEFAULT_CONFIG) to stop the variance runaway
#              that under-fits the mean. This is the current main run.
# kong_all   = plain Kong NLL on all cells (no beta-NLL) — the under-fit baseline.
# normal_only= label-defined ablation: optimise only on normal target/history
#              cells (NOT Kong: it holds rare out of the loss).
EXPERIMENT = 'beta_nll'   # 'beta_nll' | 'kong_all' | 'baseline' | 'exp1' | 'exp2' | 'normal_only'

_OVERRIDES = {
    'baseline': {},
    'exp1': {'epochs': 20, 'name': 'ep20'},
    'exp2': {'epochs': 20, 'n_sde_steps': 8, 'sigma_max': 2.0, 'name': 'sde8_sm2p0_ep20'},
    # Kong-faithful + beta-NLL (beta_nll=0.5 inherited from DEFAULT_CONFIG).
    'beta_nll': {'epochs': 10, 'train_normal_only': False, 'name': 'beta_nll_ep10'},
    # Plain Kong NLL, all data (beta_nll=0 to disable beta-NLL).
    'kong_all': {'epochs': 10, 'train_normal_only': False, 'beta_nll': 0.0, 'name': 'kong_all_ep10'},
    # Requires TRAIN-year anomaly scores to select normal target/history cells.
    'normal_only': {'epochs': 10, 'train_normal_only': True, 'name': 'normal_only_ep10'},
}

BASE_CONFIG = {**pipe.DEFAULT_CONFIG, **_OVERRIDES[EXPERIMENT]}

out_dir = pipe.make_out_dir(BASE_CONFIG)
run_name = pipe.make_run_name(BASE_CONFIG)
print('experiment:', EXPERIMENT)
print('config    :', BASE_CONFIG)
print('out_dir   :', out_dir)
print('run_name  :', run_name)
if Path(out_dir).exists():
    print('WARNING: out_dir already exists — a run would overwrite its files.')

## 3. Training command

In [ ]:
train_cmd = pipe.build_train_command(
    BASE_CONFIG, out_dir=out_dir, run_name=run_name,
    pvgis_dir=PVGIS_DIR, test_anomaly_scores=TEST_ANOMALY_SCORES,
    train_anomaly_scores=TRAIN_ANOMALY_SCORES, device='cuda', use_wandb=True)
print(' \\\n  '.join(train_cmd))

## 4. Run training

Set `RUN_TRAINING = True` to actually launch.

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    subprocess.run(train_cmd, check=True)
else:
    print('RUN_TRAINING is False — not launching. Command above is what would run.')

## 5. Post-hoc analysis

In [ ]:
RUN_ANALYSIS = True

analysis_cmd = pipe.build_analysis_command(out_dir, BASE_CONFIG)
print(' \\\n  '.join(analysis_cmd))
if RUN_ANALYSIS:
    subprocess.run(analysis_cmd, check=True)
else:
    print('RUN_ANALYSIS is False — not launching.')

## 6. Results & figures

Reads the CSVs the runner/analysis wrote, then shows: summary tables, the
SDE band (predictive std) by anomaly group, and the post-hoc figures
(histograms + boxplots). Monaco SDE U-Net: one total uncertainty band
(SDE-sample spread), no aleatoric/epistemic split.

In [ ]:
def _read(name):
    p = Path(out_dir) / name
    return pd.read_csv(p) if p.exists() else None

RESULT_FILES = ['metrics_global.csv','metrics_daytime.csv','metrics_by_anomaly_label.csv',
                'residual_bias_and_bin_metrics.csv','daytime_bin_summary.csv',
                'daytime_bin_anomaly_metrics.csv','uncertainty_response.csv','sharpness_overview.csv']
results = {n: _read(n) for n in RESULT_FILES}
for n, df in results.items():
    print((('OK  ' if df is not None else '--  ') + n) + (('  ' + str(df.shape)) if df is not None else ''))

In [ ]:
sharp = results['sharpness_overview.csv']
if sharp is not None:
    cols = [c for c in ['scope','count','picp','mae','rmse','mean_std','mpiw','nmpil'] if c in sharp.columns]
    display(sharp[cols])
bins = results['daytime_bin_summary.csv']
if bins is not None:
    display(bins)
unc = results['uncertainty_response.csv']
if unc is not None:
    display(unc)

In [ ]:
# SDE band (total predictive std) by anomaly group. Monaco does not split
# aleatoric/epistemic: the band is the spread of the SDE samples. Seasonal
# anomaly labels are evaluation strata only.
_pred = Path(out_dir) / 'predictions.csv'
if _pred.exists():
    _df = pd.read_csv(_pred, usecols=['anomaly_group', 'solar_irradiance_poa_target',
                                      'y_pred_std'])
    _day = _df[_df['solar_irradiance_poa_target'] > 10.0]
    display(_day.groupby('anomaly_group')[['y_pred_std']].mean())
else:
    print('predictions.csv not found — run training first.')

In [ ]:
MAX_PLOT_ROWS = 500_000
figure_paths = pipe.build_posthoc_figures(out_dir, max_plot_rows=MAX_PLOT_ROWS, random_state=1)
print('figures:', list(figure_paths))

In [ ]:
if figure_paths:
    from IPython.display import Image, display
    for path in figure_paths.values():
        display(Image(filename=str(path)))

## 7. W&B sweep (optional)

In [ ]:
sweep_parameters = {
    'n_sde_steps': {'values': [4, 6]},
    'sigma_max': {'values': [0.3, 0.5]},
    'ood_noise_std': {'values': [0.05, 0.1]},
}
sweep_config = pipe.make_sweep_config(sweep_parameters)
sweep_config

In [ ]:
CREATE_SWEEP = False  # set True to register the sweep on W&B
if CREATE_SWEEP:
    import wandb
    sweep_id = wandb.sweep(sweep_config, project=pipe.WANDB_PROJECT, entity=pipe.WANDB_ENTITY)
    print('sweep_id:', sweep_id)
    print('wandb agent ' + pipe.WANDB_ENTITY + '/' + pipe.WANDB_PROJECT + '/' + str(sweep_id))
else:
    print('Set CREATE_SWEEP=True to register. The agent runs the program in sweep_config:')
    print('  ' + pipe.SWEEP_MEMBER_SCRIPT)
    print('  wandb agent ' + pipe.WANDB_ENTITY + '/' + pipe.WANDB_PROJECT + '/<sweep_id>')

## 8. Log post-hoc results to W&B (optional)

Adds post-hoc scalars, figures and a compact artifact to the matching W&B run.
The sweep wrapper does this automatically per run.

In [ ]:
LOG_TO_WANDB = True
if LOG_TO_WANDB:
    import wandb
    run = pipe.init_wandb_run_for_out_dir(wandb, out_dir, run_name=run_name)
    try:
        posthoc = pipe.log_posthoc_to_wandb(
            wandb, run, out_dir,
            figure_paths=figure_paths if 'figure_paths' in dir() else None,
        )
        print(posthoc['summary'])
        print('posthoc artifact uploaded:', posthoc['artifact_uploaded'])
    finally:
        run.finish()
else:
    print('LOG_TO_WANDB is False. posthoc summary preview:')
    if Path(out_dir, 'sharpness_overview.csv').exists():
        print(pipe.read_posthoc_summary(out_dir))